# 03 — Representation Features (D-02, canonical)

> **Phase 3 · SSOT §12.2 D-02 Representation Features**
> **Canonical artifact**: `data/registry/REP_FEATURES_v002.parquet`

> **Lineage authority**: Claude-B independent forensic adjudication
> `441d5802bfebe178fd220d08b653c60dfad17faf`
> `ssot/2026-08-17_1730_KOEN_TP_G2_G3_G4_FINAL_ADJUDICATION.md`
> `G2_REPRESENTATION_INTEGRITY_PASS` · `G3_TOKENIZER_INTEGRITY_PASS` · `G4_MORPHOLOGY_INTEGRITY_PASS`
> `MEASUREMENT_FOUNDATION_CLOSED_THROUGH_G4`
>
> Every artifact identity below is pinned to values Claude-B recomputed from the physical Parquet,
> not to values this pipeline reported about itself.


## Scope

This notebook establishes that the D-02 surface representation layer of the final cohort conforms to
SSOT §12.2 and is the artifact the downstream phases consume. It covers the full variable set —
Unicode length, lexical length, byte, whitespace, script, mixing, special expression, provenance —
and the pair-level length quantities.

Morphology-derived variables listed in the §12.2 table are **not** computed here; they are D-03 and
belong to `04_morphology_features.ipynb`.

## Lineage of this layer

```
PAIR_REGISTRY_v002        D-01/P2 accepted cohort, 5,652,925 rows
        │
        ├─ REP_FEATURES_v001      HISTORICAL, 47 columns, SUPERSEDED
        │                         missing the §12.2 lexical length group
        ↓
REP_FEATURES_v002         CANONICAL, 49 columns, N = 3,835,988
```

`REP_FEATURES_v001` is retained as provenance and is **never read as current evidence**. The v002
artifact was produced by carrying every v001 column through an exact `pair_id` join and computing
only the two missing native fields, so the 47 original features were never recomputed.

## Fail-closed lineage

This notebook **validates and reuses**; it does not rebuild. Canonical inputs are asserted through
`tokenization_premium.lineage.assert_canonical_artifact`, which checks path, SHA-256, column count,
row count and identifier uniqueness, and raises `CanonicalArtifactIdentityMismatch` on any
difference. There is no `if exists()` guard and no descent to a pilot, synthetic, or earlier
version: a missing or altered artifact stops the notebook rather than silently changing what is
being reported.

## 0 — Environment and rebuild gate

In [1]:
from __future__ import annotations

import json

import duckdb
import pandas as pd

from tokenization_premium.lineage import (
    ADJUDICATION_COMMIT,
    ADJUDICATION_DOC,
    CANONICAL_ARTIFACTS,
    CANONICAL_PAIR_SET_MD5,
    CanonicalArtifactIdentityMismatch,
    assert_canonical_artifact,
    describe_historical,
)
from tokenization_premium.paths import PROJECT_ROOT

pd.set_option("display.width", 200, "display.max_columns", 50)


def connect() -> duckdb.DuckDBPyConnection:
    """모든 전집단 질의는 DuckDB pushdown으로 처리해 메모리를 bounded 상태로 유지한다."""
    con = duckdb.connect()
    con.execute("SET memory_limit='5GB'")
    con.execute("SET threads=8")
    con.execute("SET preserve_insertion_order=false")
    spill = PROJECT_ROOT / ".runtime" / "canonical-nb" / "duckdb-spill"
    spill.mkdir(parents=True, exist_ok=True)
    con.execute(f"SET temp_directory='{spill.as_posix()}'")
    return con


CON = connect()
print(f"lineage authority : {ADJUDICATION_COMMIT[:12]}  {ADJUDICATION_DOC}")

lineage authority : 441d5802bfeb  ssot/2026-08-17_1730_KOEN_TP_G2_G3_G4_FINAL_ADJUDICATION.md


In [2]:
REBUILD_CANONICAL_ARTIFACT = False   # Director-authorized rebuild only; Run All must never regenerate

if REBUILD_CANONICAL_ARTIFACT:
    raise RuntimeError(
        "REBUILD_CANONICAL_ARTIFACT=True는 Director 승인 실행에서만 사용한다. "
        "기본 Run All은 canonical artifact를 재생성하지 않고 검증·재사용만 한다."
    )
print("rebuild gate: DISABLED (validate/reuse only)")

rebuild gate: DISABLED (validate/reuse only)


## 1 — Canonical input identity (fail-closed)

Both the D-01/P2 registry and the D-02 canonical artifact are asserted before anything is read from
them. The pair-set hash is the same statistic Claude-B used, so agreement here is agreement with the
adjudication rather than with this pipeline's own bookkeeping.

In [3]:
PAIR_REGISTRY = assert_canonical_artifact("PAIR_REGISTRY_v002", con=CON)
REP = assert_canonical_artifact("REP_FEATURES_v002", verify_pair_set=True, con=CON)

for confirmed in (PAIR_REGISTRY, REP):
    print(f"{confirmed['name']:24s} {confirmed['identity']}")
    print(f"   sha256 {confirmed['sha256']}")
    print(f"   rows   {confirmed['row_count']:>9,}   columns {confirmed['column_count']:>3}"
          f"   distinct {confirmed['id_column']} {confirmed['distinct_id']:,}")
print(f"\npair-set md5 {REP['pair_set_md5']}  (Claude-B: {CANONICAL_PAIR_SET_MD5})")
assert REP["pair_set_md5"] == CANONICAL_PAIR_SET_MD5

REP_REL = CANONICAL_ARTIFACTS["REP_FEATURES_v002"].path.as_posix()
R = f"read_parquet('{REP_REL}')"
N = REP["row_count"]

PAIR_REGISTRY_v002       CANONICAL_ARTIFACT_IDENTITY_VERIFIED
   sha256 95f523d11b0e8fcfd761dee949f082e9b4590b919801441fbcfa3426010bec52
   rows   5,652,925   columns  69   distinct pair_id 5,652,925
REP_FEATURES_v002        CANONICAL_ARTIFACT_IDENTITY_VERIFIED
   sha256 dfae8e01cd3fe2ca949d8754678e508203ad1a7aa6abea418008a33ac650d309
   rows   3,835,988   columns  49   distinct pair_id 3,835,988

pair-set md5 d9660d654ee449e4d0c23a0070225274  (Claude-B: d9660d654ee449e4d0c23a0070225274)


## 2 — Historical provenance (not current evidence)

`REP_FEATURES_v001` is recorded so the lineage is reconstructible. It is deliberately not opened for
any measurement in this notebook.

In [4]:
historical = describe_historical("REP_FEATURES_v001")
print(json.dumps(historical, ensure_ascii=False, indent=2))
print("\nThis artifact is provenance only. No cell below reads it.")

{
  "relative_path": "data/registry/REP_FEATURES_v001.parquet",
  "sha256": "335c20deacb355dc1ca147547ae18e1a5ab1508fe5272508b62ee04ddf819e45",
  "status": "HISTORICAL_SUPERSEDED_BY_REP_FEATURES_v002",
  "reason": "SSOT §12.2 lexical length 변수군(ko_eojeol_count/en_word_count) 부재",
  "superseded_by": "REP_FEATURES_v002",
  "present_locally": "True",
  "NOT_CURRENT_EVIDENCE": "true"
}

This artifact is provenance only. No cell below reads it.


## 3 — SSOT §12.2 field-group conformance

The §12.2 table names nine variable groups. Eight are D-02-native and must be present here; the
morphology group is D-03 and must be absent from this artifact.

In [5]:
SIDES = ("ko", "en")
GROUPS = {
    "Unicode length": ["codepoint_count", "grapheme_count"],
    "lexical length": ["ko_eojeol_count", "en_word_count"],
    "byte": ["utf8_bytes", "bytes_per_codepoint", "bytes_per_grapheme"],
    "whitespace": ["whitespace_count", "whitespace_density", "space_run_count"],
    "script": ["hangul_share", "latin_share", "digit_share", "punctuation_share",
               "symbol_other_share"],
    "mixing": ["script_type_count", "script_switch_count"],
    "special expression": ["url_flag", "email_flag", "emoji_flag", "code_like_flag"],
}
PROVENANCE = ["feature_extractor_version", "unicode_version", "grapheme_implementation",
              "config_sha256"]
PAIR_LEVEL = ["pair_codepoint_ratio", "pair_grapheme_ratio", "pair_byte_ratio", "pair_codepoint_diff"]

present = {row[0] for row in CON.execute(f"DESCRIBE SELECT * FROM {R}").fetchall()}
rows = []
for group, fields in GROUPS.items():
    expected = fields if group == "lexical length" else [f"{s}_{f}" for s in SIDES for f in fields]
    missing = [f for f in expected if f not in present]
    rows.append({"§12.2 group": group, "expected": len(expected),
                 "present": len(expected) - len(missing), "missing": ", ".join(missing) or "—"})
rows.append({"§12.2 group": "provenance", "expected": len(PROVENANCE),
             "present": sum(f in present for f in PROVENANCE),
             "missing": ", ".join(f for f in PROVENANCE if f not in present) or "—"})
rows.append({"§12.2 group": "pair-level length", "expected": len(PAIR_LEVEL),
             "present": sum(f in present for f in PAIR_LEVEL),
             "missing": ", ".join(f for f in PAIR_LEVEL if f not in present) or "—"})
conformance = pd.DataFrame(rows)
print(conformance.to_string(index=False))

morphology_leak = [c for c in present if any(k in c for k in
                   ("morpheme", "particle_count", "ending_count", "deriv_affix"))]
print(f"\nmorphology fields in D-02 (must be empty, they are D-03): {morphology_leak or '[]'}")
assert not conformance["missing"].ne("—").any(), "SSOT §12.2 field group missing"
assert not morphology_leak
print("SSOT_12_2_FIELD_CONFORMANCE = PASS")

       §12.2 group  expected  present missing
    Unicode length         4        4       —
    lexical length         2        2       —
              byte         6        6       —
        whitespace         6        6       —
            script        10       10       —
            mixing         4        4       —
special expression         8        8       —
        provenance         4        4       —
 pair-level length         4        4       —

morphology fields in D-02 (must be empty, they are D-03): []
SSOT_12_2_FIELD_CONFORMANCE = PASS


## 4 — Lexical length, the v002 delta

`ko_eojeol_count` and `en_word_count` are the only fields v002 adds. The segmentation rule splits
`*_text_analysis` on Unicode whitespace and counts non-empty segments, with no normalisation, no
punctuation stripping, no NFKC and no lowercasing.

The rule reuses the same `regex` VERSION1 whitespace class already frozen in v001's classifier, so a
strong internal check is available: for text with no leading or trailing whitespace,
`eojeol_count == space_run_count + 1` must hold exactly. That identity is verified over the whole
population below, which is why the counts can be trusted without re-deriving them from the source
text here.

In [6]:
from tokenization_premium.representation import (
    REPRESENTATION_V2_LEXICAL_CONFIG,
    REPRESENTATION_V2_LEXICAL_CONFIG_SHA256,
)

print(json.dumps(REPRESENTATION_V2_LEXICAL_CONFIG, ensure_ascii=False, indent=1))
print(f"\nlexical rule sha256 {REPRESENTATION_V2_LEXICAL_CONFIG_SHA256}")

lex = CON.execute(f"""
SELECT
  sum((ko_eojeol_count IS NULL)::INT)                    AS ko_null,
  sum((en_word_count IS NULL)::INT)                      AS en_null,
  min(ko_eojeol_count) AS ko_min, max(ko_eojeol_count) AS ko_max, avg(ko_eojeol_count) AS ko_mean,
  min(en_word_count)   AS en_min, max(en_word_count)   AS en_max, avg(en_word_count)   AS en_mean,
  sum((ko_eojeol_count <= 0)::INT)                       AS ko_nonpositive,
  sum((en_word_count   <= 0)::INT)                       AS en_nonpositive,
  sum((ko_eojeol_count > ko_codepoint_count)::INT)       AS ko_exceeds_codepoints,
  sum((en_word_count   > en_codepoint_count)::INT)       AS en_exceeds_codepoints,
  sum((ko_eojeol_count <> ko_space_run_count + 1)::INT)  AS ko_identity_violation,
  sum((en_word_count   <> en_space_run_count + 1)::INT)  AS en_identity_violation
FROM {R}""").fetchdf().iloc[0]
for k, v in lex.items():
    print(f"  {k:24s} {v:,.4f}" if isinstance(v, float) else f"  {k:24s} {int(v):,}")

assert int(lex["ko_null"]) == int(lex["en_null"]) == 0
assert int(lex["ko_nonpositive"]) == int(lex["en_nonpositive"]) == 0, "HARD_INVALID: count <= 0"
assert int(lex["ko_identity_violation"]) == int(lex["en_identity_violation"]) == 0
print("\nLEXICAL_LENGTH_CONFORMANCE = PASS")

{
 "rule_version": "rep_lexical_v002",
 "spec_ref": "SSOT §12.2 lexical length (ko_eojeol_count, en_word_count)",
 "decision_id": "RD-20260817-D02D03-CONFORMANCE-01",
 "ko_eojeol_count": "ko_text_analysis를 Unicode whitespace로 분리한 non-empty orthographic segment 수",
 "en_word_count": "en_text_analysis를 Unicode whitespace로 분리한 non-empty surface segment 수",
 "split_pattern": "\\s+",
 "split_engine": "regex==2026.7.19 VERSION1 (Unicode whitespace)",
 "prohibited": [
  "source text normalization or collapsing before counting",
  "punctuation stripping",
  "NFKC folding",
  "lowercasing"
 ],
 "hard_invalid_rule": "accepted final cohort에서 count == 0 이면 HARD_INVALID"
}

lexical rule sha256 c1dc9dc7e6410819ba2c508c015a110a4e37eb7288ce050b42bf9caa46945b8c
  ko_null                  0.0000
  en_null                  0.0000
  ko_min                   1.0000
  ko_max                   158.0000
  ko_mean                  10.9514
  en_min                   1.0000
  en_max                   331.0000
  

## 5 — Cohort integrity

The G1 human adjudication excluded exactly 25 individually-confirmed pairs. Their absence is checked
against the adjudication document itself rather than against a copied list.

In [7]:
import re

adjudication = (PROJECT_ROOT / "ssot/HumanLebeled/KOEN_G1_HUMAN_AUDIT_FINAL_ADJUDICATION.md"
                ).read_text(encoding="utf-8")
excluded = sorted(set(re.findall(r"pair_[0-9a-f]{64}", adjudication)))
CON.execute("CREATE OR REPLACE TEMP TABLE excluded(pair_id VARCHAR)")
CON.executemany("INSERT INTO excluded VALUES (?)", [(pid,) for pid in excluded])
remaining = CON.execute(f"SELECT count(*) FROM {R} JOIN excluded USING (pair_id)").fetchone()[0]

print(f"G1-excluded pair_ids parsed from the adjudication : {len(excluded)}")
print(f"still present in REP_FEATURES_v002               : {remaining}")
print(f"row count                                        : {N:,}")
assert len(excluded) == 25 and remaining == 0
print("\nCOHORT_INTEGRITY = PASS")

G1-excluded pair_ids parsed from the adjudication : 25
still present in REP_FEATURES_v002               : 0
row count                                        : 3,835,988

COHORT_INTEGRITY = PASS


## 6 — Distribution evidence

Aggregate profile of the layer, computed by pushdown over the full population. This is descriptive
only; no inference is drawn here.

In [8]:
FEATURES = ["ko_codepoint_count", "en_codepoint_count", "ko_eojeol_count", "en_word_count",
            "ko_utf8_bytes", "en_utf8_bytes", "ko_bytes_per_codepoint", "en_bytes_per_codepoint",
            "ko_hangul_share", "en_latin_share", "ko_script_type_count",
            "pair_codepoint_ratio", "pair_byte_ratio", "pair_codepoint_diff"]
sel = ", ".join(
    f"min({f}) AS \"{f}|min\", quantile_cont({f}, 0.25) AS \"{f}|p25\", "
    f"median({f}) AS \"{f}|median\", quantile_cont({f}, 0.75) AS \"{f}|p75\", "
    f"quantile_cont({f}, 0.99) AS \"{f}|p99\", max({f}) AS \"{f}|max\", avg({f}) AS \"{f}|mean\""
    for f in FEATURES)
raw = CON.execute(f"SELECT {sel} FROM {R}").fetchdf().iloc[0]
profile = pd.DataFrame(
    [{"feature": f, **{stat: float(raw[f"{f}|{stat}"])
                       for stat in ("min", "p25", "median", "p75", "p99", "max", "mean")}}
     for f in FEATURES]).set_index("feature")
profile.round(4)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min,p25,median,p75,p99,max,mean
feature,,,,,,,
ko_codepoint_count,1.0000,23.0000,36.0000,66.0000,122.0000,684.0000,45.6974
en_codepoint_count,1.0000,46.0000,74.0000,146.0000,298.0000,2208.0000,100.3073
ko_eojeol_count,1.0000,6.0000,9.0000,16.0000,29.0000,158.0000,10.9514
en_word_count,1.0000,9.0000,13.0000,24.0000,47.0000,331.0000,16.8564
ko_utf8_bytes,1.0000,56.0000,89.0000,158.0000,292.0000,1538.0000,109.9111
en_utf8_bytes,1.0000,46.0000,74.0000,146.0000,298.0000,2208.0000,100.3240
ko_bytes_per_codepoint,1.0000,2.3548,2.4500,2.5000,2.6250,3.0000,2.4125
en_bytes_per_codepoint,1.0000,1.0000,1.0000,1.0000,1.0000,2.5088,1.0002
ko_hangul_share,0.0000,0.6765,0.7246,0.7500,0.8125,1.0000,0.7061


## 7 — Canonical summary

In [9]:
summary = {
    "notebook": "notebooks/03_representation_features.ipynb",
    "phase": "Phase 3 — D-02 Representation Features",
    "canonical_artifact": {k: REP[k] for k in
                           ("name", "path", "sha256", "row_count", "column_count", "pair_set_md5")},
    "historical": {"REP_FEATURES_v001": historical["status"]},
    "ssot_12_2_field_conformance": "PASS",
    "lexical_length_conformance": "PASS",
    "cohort_integrity": "PASS",
    "lexical_rule_sha256": REPRESENTATION_V2_LEXICAL_CONFIG_SHA256,
    "gate": "G2_REPRESENTATION_INTEGRITY_PASS (adjudicated by Claude-B at "
            f"{ADJUDICATION_COMMIT[:12]}; this notebook reproduces the evidence, it does not "
            "re-adjudicate the gate)",
    "rebuild_performed": False,
}
print(json.dumps(summary, ensure_ascii=False, indent=2))
CON.close()

{
  "notebook": "notebooks/03_representation_features.ipynb",
  "phase": "Phase 3 — D-02 Representation Features",
  "canonical_artifact": {
    "name": "REP_FEATURES_v002",
    "path": "data/registry/REP_FEATURES_v002.parquet",
    "sha256": "dfae8e01cd3fe2ca949d8754678e508203ad1a7aa6abea418008a33ac650d309",
    "row_count": 3835988,
    "column_count": 49,
    "pair_set_md5": "d9660d654ee449e4d0c23a0070225274"
  },
  "historical": {
    "REP_FEATURES_v001": "HISTORICAL_SUPERSEDED_BY_REP_FEATURES_v002"
  },
  "ssot_12_2_field_conformance": "PASS",
  "lexical_length_conformance": "PASS",
  "cohort_integrity": "PASS",
  "lexical_rule_sha256": "c1dc9dc7e6410819ba2c508c015a110a4e37eb7288ce050b42bf9caa46945b8c",
  "gate": "G2_REPRESENTATION_INTEGRITY_PASS (adjudicated by Claude-B at 441d5802bfeb; this notebook reproduces the evidence, it does not re-adjudicate the gate)",
  "rebuild_performed": false
}


---

`REP_FEATURES_v002` is the canonical D-02 artifact for all downstream phases. `REP_FEATURES_v001` is
historical and must not be read as current evidence.